# Chapter 15: Build Your Own Lift Engine

This notebook runs every piece of code in the chapter, in order, with the chapter's text around it. Read the chapter on the site: [Build Your Own Lift Engine](https://books.sugiv.fyi/lift/lift-engine.html). Everything is simulated from seeded code in `sim/`, so a run gives the same numbers as the book.

In [1]:
# Run this cell first. In Google Colab it fetches the companion repo; anywhere else, run the
# notebook from a clone of it. Either way it moves to the repo's root so `sim` can be imported.
import os
import pathlib
import sys

if "google.colab" in sys.modules:
    if not pathlib.Path("/content/lift-not-luck").exists():
        !git clone -q --depth 1 https://github.com/sugix/lift-not-luck /content/lift-not-luck
    %pip install -q pingouin
candidates = [pathlib.Path.cwd(), *pathlib.Path.cwd().parents, pathlib.Path("/content/lift-not-luck")]
root = next(p for p in candidates if (p / "sim" / "parlo.py").is_file())
os.chdir(root)
sys.path.insert(0, str(root))

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
def glue(name, value, display=True):
    """In the book, glue() puts a computed number into the text. Here the numbers are already
    written into the text, so it only shows the value when asked to."""
    if display:
        from IPython.display import display as show
        show(value)

from sim import engine as G, voicemode as V, style

style.apply()
pd.set_option("display.precision", 4)


def glue_num(name, value):
    glue(name, float(value), display=False)

Chapter 14 ended with a to-do list for Juno's team: before the review policy can launch, someone has to assign learners consistently, check the split, read the decision metric (the one number the launch rests on) and the cost guardrail (a number the team has promised not to make worse), correct for every other metric anyone glances at, and decide. Each item is a tool from an earlier chapter. Done by hand, in a fresh notebook, for every test, one of them gets skipped sooner or later, and nobody notices which.

Here's the finding first. Every one of those tools fits in about **two hundred lines of Python**, [`sim/engine.py`](https://github.com/sugix/lift-not-luck/blob/main/sim/engine.py), and the result is an engine you can run on every experiment. Given one table with a row per learner, it checks the split, estimates every metric on the right unit (the thing the coin was flipped for, here the learner), tests the guardrails, corrects for many metrics and writes a decision memo. Run on Juno's new **Voice Mode** test, the memo says the feature works, lessons per learner up 2.7%, and also that it shouldn't ship (be released to everyone) yet: cost per lesson is up 14%, against a promised limit of 10%. And because the engine is code, it can be tested like code. Across 150 A/A tests (tests where both groups get exactly the same thing, so any "win" is a false alarm) and twelve metrics, its false-alarm rate is 4.0% against the promised 5%, and its 95% confidence intervals (the range of lifts that fit the data) cover the true lift 97% of the time.

This chapter is the book's capstone. It doesn't introduce a new method. It assembles the ones you already have into a single pipeline, and at each step it says which chapter the piece came from.

You may never need to build your own. GrowthBook, Statsig and Eppo (commercial experimentation platforms) and the in-house platforms at large companies do all of this and more. But building a small one, once, is the best way I know to understand what those platforms are doing, to sanity-check them, and to analyse the experiments they don't cover: agent pipelines, offline evaluations (scoring an agent on a fixed set of test questions before any learner sees it), one-off tests run from a notebook.

> **In this chapter**
>
> - The shape every experimentation platform shares, from assignment to decision, and which chapter taught each step.
> - Hash-based assignment: a coin that gives the same learner the same answer every time.
> - Metrics declared as configuration, so the decision metric is chosen before the data arrives.
> - A readout that refuses to run when the split is wrong, and a memo that blocks a launch when a guardrail fails.
> - Testing the engine itself with A/A tests and coverage checks, the way you'd unit-test any code (check it automatically against cases where you know the right answer).
> - What a full platform adds on top, and what to ask of any tool you buy.

> **Note**
>
> The engine is [`sim/engine.py`](https://github.com/sugix/lift-not-luck/blob/main/sim/engine.py) and the demo experiment is [`sim/voicemode.py`](https://github.com/sugix/lift-not-luck/blob/main/sim/voicemode.py): Juno Voice Mode, where learners in the treatment group can talk to Juno instead of typing. Both are short enough to read in one sitting, which is the point. The simulator is seeded, so you get exactly the numbers on this page.

## The shape of an engine

Think of an experiment as a parcel going through a sorting office. It's labelled on the way in, weighed, checked for damage, and only then routed. Skip the damage check and you deliver a broken parcel with total confidence. Every experimentation platform has the same conveyor belt, whatever it calls the stations:

| Step | Question it answers | Taught in |
|---|---|---|
| **Assign** | Who's in which group, consistently? | [The Lift That Wasn't: Why We Randomise](https://books.sugiv.fyi/lift/lift-that-wasnt.html), [Designing the Test: How Big, How Long, What to Count](https://books.sugiv.fyi/lift/design-test.html) |
| **Log exposure** | Who actually saw their group, and when? | [Don't Trust It Yet: Checks Before You Read a Result](https://books.sugiv.fyi/lift/trust-checks.html) |
| **Aggregate per unit** | One row per learner, the unit you randomised | [Designing the Test: How Big, How Long, What to Count](https://books.sugiv.fyi/lift/design-test.html) |
| **Check** | Is the split what we asked for, everywhere? | [Don't Trust It Yet: Checks Before You Read a Result](https://books.sugiv.fyi/lift/trust-checks.html) |
| **Estimate** | Lift and uncertainty, with CUPED and ratio metrics | [Designing the Test: How Big, How Long, What to Count](https://books.sugiv.fyi/lift/design-test.html), [Faster Answers: CUPED and Peeking Safely](https://books.sugiv.fyi/lift/faster-answers.html) |
| **Guard** | Did anything we promised not to break, break? | [Designing the Test: How Big, How Long, What to Count](https://books.sugiv.fyi/lift/design-test.html) |
| **Correct** | Many metrics, one false-alarm budget | [Many Metrics, Long-Run Truth](https://books.sugiv.fyi/lift/many-metrics.html) |
| **Decide** | Ship, don't ship, or keep going | [Reading the Result: P-values, Chance to Win and Expected Loss](https://books.sugiv.fyi/lift/reading-result.html) |

The engine covers every step except logging and per-learner aggregation, which live in your app and your data warehouse (the company's central store of data). The rest of the chapter walks down the belt, one station per section.

## Step 1: assignment you can recompute

Chapter 1 ([The Lift That Wasn't: Why We Randomise](https://books.sugiv.fyi/lift/lift-that-wasnt.html)) showed why a coin should decide who gets a change: it balances everything, measured or not. The engineering question is how to flip that coin in a live product.

Calling `random()` when a request comes in works once. It breaks the second time the same learner shows up, because they might land in the other group, and a learner who sees both versions belongs to neither. Real systems **hash** instead. A hash function turns any text into a long, scrambled number; the same text always gives the same number, and similar texts give unrelated numbers. Combine the experiment's name (the **salt**) with the learner's id, hash it, and scale the result to a number between 0 and 1. Below 0.5 is treatment; above is control. It's a coin that remembers: the same learner always gets the same answer for the same experiment, on any server, with no database lookup.

The cell assigns 30,000 learner ids twice, once for Voice Mode and once for an unrelated Streak Shield experiment with a different salt:

In [3]:
ids = [f"u{i:06d}" for i in range(30_000)]
arm = G.assign(ids, salt="juno-voice-mode")       # hash of "juno-voice-mode:<id>", split at 0.5; the code calls each group an arm
other = G.assign(ids, salt="streak-shield-v3")    # a second experiment, a different salt
print(f"share in treatment: {(arm == 'treatment').mean():.2%}")
print(f"same learner, asked again: {G.assign(ids[:3], 'juno-voice-mode')} == {arm[:3]}")
pd.crosstab(pd.Series(arm, name="Voice Mode"), pd.Series(other, name="Streak Shield v3"))

share in treatment: 49.99%
same learner, asked again: ['control' 'control' 'treatment'] == ['control' 'control' 'treatment']


Streak Shield v3,control,treatment
Voice Mode,,
control,7413,7591
treatment,7325,7671


Three properties, all visible above. The split is **balanced**: half the learners in each group. It's **consistent**: asking again gives the same answer. And a different salt gives an **independent** split: the crosstab (a table counting learners in each combination of groups) has roughly equal counts in all four cells, so Voice Mode learners are spread evenly across both Streak Shield groups. That last one matters. If both experiments used the same salt, every Voice Mode learner would also be in the same Streak Shield group, and the two effects would be impossible to separate.

The coin is in place. Next, decide what you'll measure, before any data arrives.

## Step 2: metrics as configuration

<details>
<summary><b>Refresher: decision metrics and guardrails</b></summary>

A **decision metric** is the one number the ship decision rests on, chosen before the test. A **guardrail** is a metric you promise not to harm by more than a stated **margin**, such as "latency may not rise by more than 10%". **Secondary metrics** are everything else you'll look at, and they don't decide anything on their own. Chapter 2 ([Designing the Test: How Big, How Long, What to Count](https://books.sugiv.fyi/lift/design-test.html)) wrote all of this down as a one-page design before the test started.

</details>

In the engine, metrics are declared once, as data, with the details that decide how each is analysed: what's summed per learner, whether it's a ratio (and of what), which pre-test column CUPED can use, the metric's role and, for guardrails, the margin.

In [4]:
metrics = [
    G.Metric("lessons per learner", "lessons", covariate="pre_lessons", role="decision"),
    G.Metric("latency per turn", "latency_total_s", "turns", role="guardrail",
             higher_is_better=False, margin=0.10),           # may not get more than 10% slower
    G.Metric("cost per lesson", "cost_usd", "lessons", role="guardrail",
             higher_is_better=False, margin=0.10),           # may not get more than 10% dearer
    G.Metric("completion per lesson", "lessons_completed", "lessons"),
    G.Metric("minutes studied", "minutes"),
] + [G.Metric(name.replace("_", " "), name) for name in V.SECONDARY_NULL]   # seven metrics Voice Mode doesn't touch

voice = G.Experiment("Juno Voice Mode", metrics, segments=["platform"])

This list *is* Chapter 2's one-page design, in code. Writing it before the test starts, and committing it (saving it in the code's version history, where every change is dated), is what stops anyone choosing the decision metric after seeing the results. Notice that the agent metrics, latency and cost, sit right next to the product metrics. For an agent feature they're not an afterthought; they're half the decision.

The seven metrics in `V.SECONDARY_NULL` are there on purpose: the simulator gives Voice Mode no effect on any of them. They play the part of the dashboard tiles (the small charts on the screen of numbers a team watches) every team glances at, and they'll test whether the engine raises false alarms.

## Step 3: check before you read

<details>
<summary><b>Refresher: sample ratio mismatch</b></summary>

If you asked for a 50/50 split and got 49/51 on a large test, something other than the coin decided who's in your data: a crash, a redirect, a logging bug. A chi-square test says how surprising the split is if the coin were fair. When it fails, the two groups are no longer comparable, and no metric can be trusted until you find the cause. Chapter 3 ([Don't Trust It Yet: Checks Before You Read a Result](https://books.sugiv.fyi/lift/trust-checks.html)) made it the first check to run, before reading any metric.

</details>

The engine refuses to read any metric until the sample ratio checks pass, overall and inside every segment you listed (a slice of learners; here, by platform: iOS, Android or web). A check fails when its p-value (the chance of a split at least this lopsided if the coin were fair) is below 0.001: strict, because the engine runs many of these checks and a false stop is costly too.

To see the refusal, the cell plants Chapter 3's bug: the new voice screen crashes on some Android phones before exposure is logged (before the app records that the learner saw the new screen), so those learners silently vanish from the treatment group.

In [5]:
broken = V.simulate(arm=arm, crash_share=0.08)   # 8% of Android treatment learners lost from the logs
print(G.decide(broken, voice))
G.srm_table(broken, voice).round(4)

STOP: sample ratio mismatch in platform = Android. Find the cause before reading any metric.


,users,treatment share,p_value,fails (p < 0.001)
slice,,,,
overall,29540,0.4921,0.0065,False
platform = Android,11493,0.4818,0.0001,True
platform = iOS,13538,0.4958,0.3272,False
platform = web,4509,0.5072,0.3330,False


The memo stops at the first line. Look at the table to see why checking by segment matters. Overall, the split is only slightly off, and its p-value doesn't cross the engine's line. Inside Android it's clearly broken, while iOS and web look fine. An overall check alone would have let this test through. In this simulation the lost learners are a random slice of Android users, but in real life the phones that crash rarely are (older, cheaper, slower), and the lift would be biased in a direction nobody could predict. Fix the crash, rerun, and the engine moves on.

## Step 4: the readout

With a clean split, the engine reads every metric. The cell simulates the fixed test on the same hashed assignment and prints the full readout:

In [6]:
data = V.simulate(arm=arm)                       # the same 30,000 learners, no crash
report = G.readout(data, voice)
cols = ["role", "control", "treatment", "lift", "ci_low", "ci_high", "p_value", "p_bh", "guardrail"]
report[cols].style.format({"control": "{:.3f}", "treatment": "{:.3f}", "lift": "{:+.1%}",
                           "ci_low": "{:+.1%}", "ci_high": "{:+.1%}", "p_value": "{:.4f}",
                           "p_bh": "{:.3f}"}, na_rep="")

,role,control,treatment,lift,ci_low,ci_high,p_value,p_bh,guardrail
metric,,,,,,,,,
lessons per learner,decision,8.711,8.943,+2.7%,+1.2%,+4.1%,0.0003,,
latency per turn,guardrail,1.472,1.551,+5.4%,+4.2%,+6.6%,0.0000,,pass
cost per lesson,guardrail,0.006,0.007,+14.4%,+13.2%,+15.6%,0.0000,,fail
completion per lesson,secondary,0.737,0.750,+1.8%,+1.1%,+2.5%,0.0000,0.000,
minutes studied,secondary,19.188,20.358,+6.1%,+3.6%,+8.5%,0.0000,0.000,
streak days,secondary,1.530,1.552,+1.4%,-1.1%,+4.0%,0.2751,0.563,
vocab reviews,secondary,2.302,2.329,+1.2%,-1.2%,+3.5%,0.3314,0.563,
shares,secondary,3.050,3.081,+1.0%,-1.2%,+3.2%,0.3753,0.563,
ratings given,secondary,3.834,3.841,+0.2%,-1.9%,+2.3%,0.8613,0.960,


Every row was computed the way an earlier chapter said it should be. One line each, with a refresher below for the details:

- **Lessons per learner** uses CUPED on pre-test lessons: subtract what each learner's history already predicted, and the noise shrinks ([Faster Answers: CUPED and Peeking Safely](https://books.sugiv.fyi/lift/faster-answers.html)).
- **Latency per turn, cost per lesson and completion per lesson** are ratio metrics, analysed per learner with the delta method, not per turn or per lesson, because the learner is what the coin assigned ([Designing the Test: How Big, How Long, What to Count](https://books.sugiv.fyi/lift/design-test.html)).
- **Guardrails** are non-inferiority tests against their 10% margins: "pass" means the data rule out a harm that big ([Designing the Test: How Big, How Long, What to Count](https://books.sugiv.fyi/lift/design-test.html)).
- **Secondary metrics** get Benjamini-Hochberg adjusted p-values, the `p_bh` column, so twelve metrics don't buy twelve chances of a false win ([Many Metrics, Long-Run Truth](https://books.sugiv.fyi/lift/many-metrics.html)). Only two survive: completion per lesson and minutes studied. The other seven moved by chance or not at all, which is exactly what the simulator built into them.

<details>
<summary><b>Refresher: CUPED and the delta method</b></summary>

**CUPED** (Controlled-experiment Using Pre-Experiment Data) uses a number you knew before the test, here each learner's lessons in the previous period, to explain away part of the noise in the outcome. The estimate stays unbiased because the coin can't have affected the past, and the confidence interval gets narrower. Chapter 5 ([Faster Answers: CUPED and Peeking Safely](https://books.sugiv.fyi/lift/faster-answers.html)) used it to roughly halve a test's length.

The **delta method** gives an honest standard error for a ratio such as completed lessons per lesson. It treats each learner as one observation with two totals (lessons taken and lessons completed), instead of pretending every lesson is independent. Chapter 2 ([Designing the Test: How Big, How Long, What to Count](https://books.sugiv.fyi/lift/design-test.html)) showed that per-lesson analysis raises false alarms several times over.

</details>

<details>
<summary><b>Refresher: non-inferiority</b></summary>

"No significant difference" on a guardrail only means the test couldn't tell. A **non-inferiority test** asks the right question: *can we rule out a harm as big as the margin?* It's a one-sided test against the margin instead of against zero. In the readout, a guardrail passes when the whole 95% interval stays on the safe side of the margin. Chapter 2 ([Designing the Test: How Big, How Long, What to Count](https://books.sugiv.fyi/lift/design-test.html)) showed why small guardrail tests pass changes they shouldn't.

</details>

<details>
<summary><b>Refresher: Benjamini-Hochberg</b></summary>

Look at twenty metrics that a change doesn't touch, and on average one of them will be "significant" at 5% by luck. Benjamini-Hochberg controls the **false discovery rate**: among the metrics you call winners, the expected share of false ones stays at or below 5%. It sorts the p-values and holds the smallest to the strictest bar, relaxing as it goes down the list. Chapter 6 ([Many Metrics, Long-Run Truth](https://books.sugiv.fyi/lift/many-metrics.html)) applied it to Juno v3's 25 rubric metrics.

</details>

How close are these estimates to the truth? The simulator can compute each metric's real lift by running a huge population, 300,000 learners, through both groups with the same random seed (the starting number that makes a simulation repeatable):

In [7]:
everyone_t = V.simulate(n=300_000, arm=np.full(300_000, "treatment"), seed=99)   # everyone gets Voice Mode
everyone_c = V.simulate(n=300_000, arm=np.full(300_000, "control"), seed=99)     # nobody does


def true_lift(m):
    if m.denominator is None:
        return everyone_t[m.numerator].mean() / everyone_c[m.numerator].mean() - 1
    rt = everyone_t[m.numerator].sum() / everyone_t[m.denominator].sum()
    rc = everyone_c[m.numerator].sum() / everyone_c[m.denominator].sum()
    return rt / rc - 1


truth = pd.Series({m.name: true_lift(m) for m in metrics})
check = report[["lift", "ci_low", "ci_high"]].assign(true_lift=truth)
check["CI covers truth"] = (check.ci_low <= check.true_lift) & (check.true_lift <= check.ci_high)
check.head(5).style.format({"lift": "{:+.1%}", "ci_low": "{:+.1%}", "ci_high": "{:+.1%}", "true_lift": "{:+.1%}"})

,lift,ci_low,ci_high,true_lift,CI covers truth
metric,,,,,
lessons per learner,+2.7%,+1.2%,+4.1%,+3.6%,True
latency per turn,+5.4%,+4.2%,+6.6%,+6.0%,True
cost per lesson,+14.4%,+13.2%,+15.6%,+15.1%,True
completion per lesson,+1.8%,+1.1%,+2.5%,+1.9%,True
minutes studied,+6.1%,+3.6%,+8.5%,+5.2%,True


In [8]:
glue_num("dec_true", 100 * truth["lessons per learner"])

All five intervals contain the true lift. None of the estimates is exactly right: lessons per learner reads 2.7% against a true 3.6%. That's Chapter 1's lesson again: one test is one draw, and the interval, not the point estimate, is the honest answer.

## Step 5: the decision memo

<details>
<summary><b>Refresher: chance to win and expected loss</b></summary>

The **chance to win** is the probability, given the data, that the change is better than what you have. The **expected loss** asks the decision question instead: *if we ship and we're wrong, how much do we lose on average?* You ship when the expected loss falls below a **threshold of caring**, an amount you'd genuinely shrug at, written down before the test. Chapter 4 ([Reading the Result: P-values, Chance to Win and Expected Loss](https://books.sugiv.fyi/lift/reading-result.html)) read one result three ways: p-value, chance to win, expected loss.

</details>

The last station turns the readout into a few sentences a product manager (the person who decides what goes into the product) can act on. Its logic is short enough to read in the source: stop on any sample ratio failure; report the decision metric with its chance to win and expected loss; block on any failed guardrail; otherwise ship when the expected loss is below the threshold of caring, or keep running if it isn't.

In [9]:
print(G.decide(data, voice))      # the whole pipeline: check, read, guard, correct, decide

Experiment: Juno Voice Mode
Decision metric 'lessons per learner': +2.7% (95% CI +1.2% to +4.1%), chance to win 100%, expected loss if shipped 0.00%.
Guardrail 'latency per turn': +5.4% (95% CI +4.2% to +6.6%), pass.
Guardrail 'cost per lesson': +14.4% (95% CI +13.2% to +15.6%), fail.
Secondary metrics moving after FDR correction: completion per lesson, minutes studied.
RECOMMENDATION: do not ship as is. Guardrail(s) not shown to be safe: cost per lesson.


In [10]:
d = report.loc["lessons per learner"]
c = report.loc["cost per lesson"]
glue_num("dec_lift", 100 * d.lift)
glue_num("cost_lift", 100 * c.lift)
glue_num("cost_lo", 100 * c.ci_low)

Voice Mode works: lessons per learner are up 2.7%, with essentially no chance it's a loss. But cost per lesson is up 14%, and even the bottom of its confidence interval (13%) is above the 10% the team promised. So the memo says **don't ship as is**.

That's the most useful kind of result, because it says what to fix. Voice turns use more tokens: transcription, then the tutor, then speech. The team can cache common phrases (generate them once and reuse them), use a smaller speech model, or reconsider the margin with the finance team, *explicitly*, and then test again. What it shouldn't do is quietly drop the guardrail because the headline metric looks good. An engine that blocks the launch in writing makes that conversation happen.

The engine gives sensible answers on one test. But one good answer doesn't prove the code is right. How would you know if it had a bug?

## Step 6: test the engine like code

<details>
<summary><b>Refresher: A/A tests</b></summary>

An **A/A test** splits learners into two groups that get exactly the same thing. Nothing changed, so any "significant" result is a false alarm, and at a 5% significance level about 5% of A/A tests should raise one. If far more do, the pipeline is broken: a wrong unit, a leaky metric, a bad split. Chapter 3 ([Don't Trust It Yet: Checks Before You Read a Result](https://books.sugiv.fyi/lift/trust-checks.html)) used A/A tests to catch a pipeline that cried wolf.

</details>

An analysis engine is code, and code has bugs. Statistical bugs are the worst kind, because they don't crash: they print plausible numbers. The test for them comes straight from Chapter 3. Run the engine on many A/A tests, where nothing changes, and check that each metric's false-alarm rate is about 5%. Then run it on tests with known effects and check that its 95% intervals contain the true lift about 95% of the time. The cell runs 150 of each, with 10,000 learners per test:

In [11]:
reps = 150
alarms = pd.DataFrame(0.0, index=[m.name for m in metrics], columns=["A/A false alarms"])
for s in range(reps):
    aa = V.simulate(n=10_000, effects={}, seed=1_000 + s)         # A/A: Voice Mode changes nothing
    alarms["A/A false alarms"] += (G.readout(aa, voice).p_value < 0.05).astype(float)
alarms /= reps

covered = pd.Series(0.0, index=[m.name for m in metrics])
for s in range(reps):
    ab = V.simulate(n=10_000, seed=5_000 + s)                     # A/B with the true effects
    r = G.readout(ab, voice)
    covered += ((r.ci_low <= truth) & (truth <= r.ci_high)).astype(float)
alarms["CI covers the true lift"] = covered / reps
alarms.style.format("{:.0%}")

,A/A false alarms,CI covers the true lift
lessons per learner,1%,98%
latency per turn,2%,91%
cost per lesson,2%,95%
completion per lesson,3%,94%
minutes studied,5%,97%
streak days,6%,98%
vocab reviews,2%,97%
shares,3%,98%
ratings given,6%,98%
settings changes,9%,99%


In [12]:
glue_num("fa_min", 100 * alarms["A/A false alarms"].min())
glue_num("fa_max", 100 * alarms["A/A false alarms"].max())
glue_num("cov_min", 100 * alarms["CI covers the true lift"].min())
glue_num("cov_max", 100 * alarms["CI covers the true lift"].max())
glue_num("fa_pooled", 100 * alarms["A/A false alarms"].mean())
glue_num("cov_pooled", 100 * alarms["CI covers the true lift"].mean())

Pooled across all twelve metrics, the false-alarm rate is 4.0% and the intervals cover the truth 97% of the time. Individual metrics range from 1% to 9% false alarms and 91% to 99% coverage. That spread is about what 150 tests give by chance alone: with true rates of 5% and 95%, anything from about 1% to 9%, and about 91% to 99%, is normal. To pin each metric down tightly, run a few thousand A/A tests overnight.

Now imagine someone "optimises" the engine by analysing ratio metrics per lesson instead of per learner. Nothing crashes, and every readout still looks reasonable. But the A/A false-alarm rate for those metrics jumps, as Chapter 2 showed, and this test catches it. **Put it in CI**, your continuous-integration checks, so it runs on every change to the engine. It takes seconds.

## What a platform adds

The engine is the statistical core: the part that decides whether a number means anything. A production platform wraps it in the things that make it safe for a whole company to use:

- **A metric layer.** Metrics defined once in SQL (the standard language for querying databases) against the warehouse, versioned and reviewed, so every team computes "lessons per learner" the same way.
- **Exposure logging and assignment services.** The hash from step 1, served at low latency (answering in milliseconds), with exposure events logged when the learner actually sees their group ([Don't Trust It Yet: Checks Before You Read a Result](https://books.sugiv.fyi/lift/trust-checks.html)).
- **Scheduled readouts and always-valid monitoring.** Daily updates with sequential tests that stay valid however often you look, so peeking (looking at results before the test is due to end) is safe and guardrails can stop a test early ([Faster Answers: CUPED and Peeking Safely](https://books.sugiv.fyi/lift/faster-answers.html)).
- **Holdouts and bandits.** Long-term holdouts ([Proving a Bandit Worked: Holdouts and Honest Numbers](https://books.sugiv.fyi/lift/proving-lift.html)) and bandit allocation ([Explore vs Exploit: Bandits from Scratch](https://books.sugiv.fyi/lift/explore-exploit.html), [Bandits in Production: Delays, Proxies and Guardrails](https://books.sugiv.fyi/lift/bandits-production.html)) built in, with propensities (the probability of each choice) logged.
- **Heterogeneity** (how the effect differs between learners). Segment breakdowns with correction, and uplift models for targeting ([Who Benefits? Uplift Models and Targeting Policies](https://books.sugiv.fyi/lift/who-benefits.html)).

The tools that do this come in two broad kinds, and it helps to know which one you're looking at. **Experimentation platforms**, such as GrowthBook, Statsig, Eppo and Optimizely, and the in-house systems at large companies, run tests and read them out for whatever you change. **Decisioning platforms** go further and make the choices themselves. JustAI, the company behind the Better Mortgage case study in Part II (JustAI, 2026), is one example for lifecycle messaging (the emails and notifications a company sends each customer over time): by its own public description, the system picks each customer's next message and its order from a shared record of customer context, tests combinations of messages, and reports lift against a holdout. That's this book's bandits and holdouts in product form: an algorithm choosing, and a holdout proving. I mention it to show where the book's methods sit inside real products, not to recommend any vendor. Whatever the tool, ask it the questions this book taught: what's the unit, where's the holdout, are propensities logged, and how are guardrails tested?

For agent teams, two additions matter most. **Log decision inputs and probabilities** for every model call, tool call and route ([Contextual Bandits: The Right Message for Each Learner](https://books.sugiv.fyi/lift/contextual-bandits.html), [Proving a Bandit Worked: Holdouts and Honest Numbers](https://books.sugiv.fyi/lift/proving-lift.html), [When You Can't Randomise: Propensity, Weighting and Double ML](https://books.sugiv.fyi/lift/observational.html)): it turns every production decision into usable evidence, even when nobody planned an experiment. And **treat offline evaluations as experiments**: an eval run over a fixed set of prompts is a paired test (both versions answer every prompt, so each prompt is its own comparison), and it deserves the same checks, units and corrections as anything else in this book.

> **In the wild**
>
> Kohavi, Tang and Xu (Kohavi et al., 2020), who between them built or ran experimentation at Microsoft, Google and LinkedIn, describe how such platforms are put together. They share this chapter's skeleton: stable hashing for assignment, a central metric definition layer, automatic sample ratio checks, and readouts that refuse to show results when a check fails.

## Pitfalls

- **Random assignment at request time.** Learners flip between groups. Hash a stable id with a per-experiment salt.
- **One salt for every experiment.** Groups line up across experiments, and effects become inseparable.
- **Reading metrics before the checks.** Make the engine refuse, as `decide` does.
- **Untested analysis code.** A/A tests are unit tests for statistics. Run them on every change.
- **Building the platform before the habits.** The hard parts are cultural: deciding metrics in advance, respecting guardrails, reporting flat results. An engine can enforce some of that; it can't create it.

> **What to take away**
>
> - Every experimentation platform runs the same belt: assign, log, aggregate, check, estimate, guard, correct, decide. Each station is a chapter of this book.
> - Hash a learner id with a per-experiment salt: consistent for each learner, independent across experiments.
> - Declare metrics, roles and margins as configuration before the test, and make the engine refuse to read anything until the split checks pass, overall and by segment.
> - A good memo can say "it works, and don't ship it": Voice Mode raised lessons by 2.7% but failed its cost guardrail.
> - Test the engine like code. A/A false alarms near 5% and coverage near 95% are its unit tests.

## Exercises

1. **A sequential guardrail.** Add Chapter 5's always-valid confidence sequence to `readout`, and make `decide` stop early when a guardrail is shown to be harmful.
2. **Uneven splits.** Run Voice Mode as a 10/90 test: pass `treatment_share=0.1` to both `G.assign` and `G.Experiment`. What changes in the readout and the sample ratio check?
3. **Break it on purpose.** Change `_arm_stats` to compute ratio metrics per lesson instead of per learner. Rerun the A/A test. Which metrics' false-alarm rates move?
4. **An agent eval.** Juno's team runs 400 fixed prompts through two prompt versions and scores each answer. Adapt the engine to a paired design, where each prompt is its own control.

## Where to go from here

This book started with a dashboard that credited Streak Shield with a lift it didn't earn, and it ends with an engine that refuses to read a test until the split is right. Everything in between is one habit, applied in different places: before you believe a number, ask what would have happened without the change, and make the data able to answer.

If you take one thing back to your team on Monday, don't make it the whole engine. Make it the smallest piece you're missing. For most teams that's one of three things: a one-page design written before the test (Chapter 2), a sample ratio check that runs before anyone sees a metric (Chapter 3), or a small random holdout behind every bandit and agent router (Chapter 10). Each takes an afternoon. Each will, sooner or later, stop you shipping something that only looked like it worked.

Then, when you can't randomise, reach for Part III, and say plainly which assumption your answer rests on. When the average is flat, look for who benefits before you delete anything. And when a result looks spectacular, look for selection first. Most real changes move metrics by a percentage point or two (the plain gap between two percentages, as in Chapter 1). Proving which ones did is the job.

Lift, not luck.

## Further reading

Kohavi, Tang and Xu (Kohavi et al., 2020) describe how experimentation platforms at Microsoft, Google and LinkedIn are built and run, including the cultural side. Deng, Knoblich and Lu (Deng et al., 2018) cover the delta method as platforms use it. Fabijan and colleagues (Fabijan et al., 2019) describe automated sample ratio checks in production.